# Notebook 03: FFT Feature Extraction

This notebook creates time-windowed traffic signals and extracts FFT-based frequency-domain features.

## Short Problem Statement

Existing deep learning-based **Intrusion Detection Systems (IDS)** often rely mainly on time-domain flow features and may miss hidden burst patterns, repeated traffic rhythms, and frequency-based attack signatures. **SentinelFlow** addresses this by using **Fast Fourier Transform (FFT)**-enhanced traffic profiling to improve deep learning-based intrusion detection on large-scale network traffic datasets.

In [ ]:
from pathlib import Path
import sys, json
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'sentinelflow_utils.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from sentinelflow_utils import *
ensure_dirs(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
cleaned_path = PROJECT_ROOT / 'data/processed/sentinelflow_cleaned_netflow.parquet'
if not cleaned_path.exists():
    cleaned_path = PROJECT_ROOT / 'data/processed/sentinelflow_analysis_dataset.parquet'
print('Loading:', cleaned_path)
df = read_table_safely(cleaned_path)
print('Shape:', df.shape)

In [ ]:
WINDOW_SECONDS = 1
SEGMENT_SIZE = 16
STRIDE = 4

signal_df, mode = make_signal_table(df, window_seconds=WINDOW_SECONDS)
print('Signal mode:', mode)
print('Signal table shape:', signal_df.shape)
signal_path = PROJECT_ROOT / 'data/processed/03_signal_table.parquet'
signal_df.to_parquet(signal_path, index=False)
print('Saved:', signal_path)
signal_df.head()

In [ ]:
baseline_df, fft_df, signal_cols = build_segment_datasets(signal_df, segment_size=SEGMENT_SIZE, stride=STRIDE)
print('Signal columns used:', signal_cols)
print('Baseline segment shape:', baseline_df.shape)
print('FFT-enhanced segment shape:', fft_df.shape)
base_path = PROJECT_ROOT / 'data/processed/03_baseline_segments.parquet'
fft_path = PROJECT_ROOT / 'data/processed/03_fft_segments.parquet'
baseline_df.to_parquet(base_path, index=False)
fft_df.to_parquet(fft_path, index=False)
print('Saved:', base_path)
print('Saved:', fft_path)

In [ ]:
segment_summary = {
    'window_seconds': WINDOW_SECONDS,
    'segment_size': SEGMENT_SIZE,
    'stride': STRIDE,
    'signal_mode': mode,
    'signal_table_shape': list(signal_df.shape),
    'baseline_shape': list(baseline_df.shape),
    'fft_shape': list(fft_df.shape),
    'signal_cols': signal_cols,
}
path = PROJECT_ROOT / 'outputs/fft_feature_summary.json'
path.write_text(json.dumps(segment_summary, indent=2), encoding='utf-8')
segment_summary